# 01 — Dataset audit and exploratory analysis

Run from the repository root. Inspect file integrity, class counts, image dimensions, and representative examples before training. The class labels come from the supplied folder names.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from src.dataset import DEFAULT_CLASSES, discover_images, audit_dataset, stratified_split
ROOT = Path('data/raw')
audit = audit_dataset(ROOT)
print('Total files:', audit['total'])
print('Counts:', audit['counts'])
print('Unreadable:', len(audit['unreadable']))
print('Dimension outliers:', len(audit['dimension_outliers']))
display(pd.DataFrame(audit['counts'].items(), columns=['stage','images']))

In [ ]:
records = discover_images(ROOT)
counts = pd.Series([DEFAULT_CLASSES[y] for _, y in records]).value_counts().reindex(DEFAULT_CLASSES, fill_value=0)
ax = counts.plot.bar(color='#6b8e23', title='Image count by ripeness stage')
ax.set_ylabel('Images'); plt.tight_layout(); plt.show()

In [ ]:
# Visual audit: sample images reveal label noise, backgrounds, lighting, and capture bias.
fig, axes = plt.subplots(len(DEFAULT_CLASSES), 5, figsize=(12, 2.5*len(DEFAULT_CLASSES)))
for row, label in enumerate(DEFAULT_CLASSES):
    examples = [p for p,y in records if DEFAULT_CLASSES[y] == label][:5]
    for col, ax in enumerate(axes[row]):
        ax.axis('off')
        if col < len(examples):
            with Image.open(examples[col]) as im: ax.imshow(im.convert('RGB'))
            if col == 0: ax.set_ylabel(label)
plt.tight_layout(); plt.show()

## Data quality and bias notes

Review the reported unreadable images and dimension outliers. Image area is only a screening signal; it is not a reason to delete files automatically. Inspect class balance and examples for backgrounds, lighting, fruit variety, camera, and collection-session effects. If near-duplicate views or multiple photos of one fruit exist, split by fruit/session to prevent leakage. Record any exclusions and their reasons here before fitting.